# Paper Selection — Semantic Screening dengan SBERT (SPECTER2)

Pipeline seleksi paper berbasis **semantic similarity** menggunakan SPECTER2
(`allenai/specter2_base`), model sentence-transformer yang dilatih khusus pada paper akademik.

**Alur:**
1. Load & prepro dasar (drop missing / duplikat)
2. **Hard AND gate** — wajib memuat istilah LLM **DAN** edukasi
3. **EDA artefak teknis** — deteksi & preview sebelum/sesudah pembersihan
4. **Statistik panjang** title+abstract vs batas token SPECTER2
5. Encoding SBERT + cosine similarity ke 21 kalimat pembanding
6. Penentuan threshold data-driven + validasi
7. Export

**Catatan penting:** preprocessing untuk SBERT **berbeda total** dari TF-IDF.
Stopword, tanda baca, dan kapitalisasi **dipertahankan** karena transformer memakainya
untuk memahami struktur kalimat. Yang dibuang hanya artefak teknis (boilerplate, mojibake).

## 0. Setup

In [ ]:
# !pip install sentence-transformers pandas numpy matplotlib kneed
import pandas as pd, numpy as np, re, html, unicodedata
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from kneed import KneeLocator

CSV_PATH  = 'IEEE__1_.csv'            # sesuaikan
OUT_PATH  = 'accepted_papers_sbert.csv'
VAL_OUT   = 'validasi_manual_sbert.csv'
MODEL_NAME = 'allenai/specter2_base'  # SPECTER2: sentence-transformer untuk paper akademik
MAX_TOKENS = 512                      # batas token SPECTER2

## 1. Load & Prepro Dasar
Hanya drop missing & duplikat. **Tidak** ada lowercasing/stopword removal di sini —
teks harus tetap natural untuk SBERT.

In [ ]:
df = pd.read_csv(CSV_PATH)
n0 = len(df)
df = df.dropna(subset=['Document Title', 'Abstract'])
n1 = len(df)
df = df.drop_duplicates(subset=['Document Title', 'Abstract']).reset_index(drop=True)
n2 = len(df)

print(f'Raw rows            : {n0}')
print(f'After drop missing  : {n1}  (removed {n0-n1})')
print(f'After drop dupes    : {n2}  (removed {n1-n2})')
print('\n=== Document Identifier (after prepro) ===')
print(df['Document Identifier'].value_counts())

# teks gabungan — mentah, belum dibersihkan
df['text_raw'] = (df['Document Title'].fillna('') + '. ' + df['Abstract'].fillna('')).astype(str)

## 2. HARD AND GATE — wajib LLM **DAN** Educational
Didaur ulang dari notebook sebelumnya. Gate ini tetap **lexical** (berbasis kata) dan berjalan
pada teks yang di-lowercase *sementara* hanya untuk pengecekan — teks asli tidak diubah.

In [ ]:
llm_terms = [
    "large language model", "language model", "pre-trained language model",
    "pretrained language model", "generative language model",
    "generative artificial intelligence", "generative ai",
    "llm", "llms", "gpt", "bert", "transformer",
]

edu_terms = [
    "education", "educational", "student", "students", "teacher", "teaching",
    "classroom", "pedagog", "school", "university", "academic", "learner",
    "tutoring", "tutor", "e-learning", "elearning", "mooc", "curriculum",
    "curricula", "assessment", "grading", "exam", "coursework", "lecture",
    "instructional", "didactic", "scholastic", "learning analytics",
    "intelligent tutoring", "educational data mining",
]

ml_learning_ctx = [
    "machine learning", "deep learning", "reinforcement learning",
    "transfer learning", "supervised learning", "unsupervised learning",
    "federated learning", "self-supervised learning", "active learning",
    "ensemble learning", "representation learning", "metric learning",
]

def has_llm(t):
    return any(term in t for term in llm_terms)

def has_edu(t):
    if any(term in t for term in edu_terms):
        return True
    if "learning" in t:                       # 'learning' standalone
        stripped = t
        for ml in ml_learning_ctx:
            stripped = stripped.replace(ml, " ")
        if "learning" in stripped:
            return True
    return False

_probe = df['text_raw'].str.lower()           # hanya untuk pengecekan gate
df['gate_llm'] = _probe.apply(has_llm)
df['gate_edu'] = _probe.apply(has_edu)
df['pass_gate'] = df['gate_llm'] & df['gate_edu']

n_before = len(df)
print('=== HARD AND GATE ===')
print(f'Sebelum gate           : {n_before}')
print(f'  mengandung LLM       : {df.gate_llm.sum()}')
print(f'  mengandung Education : {df.gate_edu.sum()}')
print(f'  LOLOS (LLM & Edu)    : {df.pass_gate.sum()}')
print(f'  DIBUANG              : {n_before - df.pass_gate.sum()}')

df = df[df['pass_gate']].reset_index(drop=True)
print(f'\nLanjut ke encoding: {len(df)} paper')

## 3. EDA — Deteksi Artefak Teknis
Sebelum membersihkan, kita **buktikan dulu** artefak apa saja yang benar-benar ada di korpus ini
dan berapa banyak. Tanpa langkah ini, pembersihan cuma tebak-tebakan.

In [ ]:
# pola artefak yang lazim di hasil scrape database akademik
artifact_patterns = {
    'copyright'      : r'(?i)(©|\(c\)|copyright)\s*\d{4}|all rights reserved|©\s*\d{4}',
    'publisher_note' : r'(?i)(published by|personal use is permitted|reprints? and permission)',
    'abstract_label' : r'(?i)^\s*abstract\s*[—\-–:]',
    'index_terms'    : r'(?i)(index terms|keywords)\s*[—\-–:]',
    'mojibake'       : r'â€™|â€œ|â€\x9d|â€"|Ã©|Ã¨|Ã¼|â€˜|Â',
    'html_entity'    : r'&[a-z]+;|&#\d+;',
    'html_tag'       : r'<[/]?(i|b|sub|sup|em|italic|bold)>',
    'citation_num'   : r'\[\d+(,\s*\d+)*\]',
    'multi_space'    : r'\s{2,}',
    'control_char'   : r'[\x00-\x08\x0b-\x1f\x7f]',
}

print('=== DETEKSI ARTEFAK ===')
hits = {}
for name, pat in artifact_patterns.items():
    mask = df['text_raw'].str.contains(pat, regex=True, na=False)
    hits[name] = mask
    pct = 100*mask.sum()/len(df)
    print(f'  {name:16s}: {mask.sum():>6} paper ({pct:5.1f}%)')

In [ ]:
# Tampilkan CONTOH NYATA tiap artefak yang ditemukan (bukti, bukan asumsi)
print('=== CONTOH ARTEFAK YANG DITEMUKAN ===\n')
for name, mask in hits.items():
    if mask.sum() == 0:
        print(f'--- {name}: tidak ditemukan ---\n'); continue
    sample = df.loc[mask, 'text_raw'].iloc[0]
    pat = artifact_patterns[name]
    m = re.search(pat, sample)
    if m:
        s, e = max(0, m.start()-70), min(len(sample), m.end()+70)
        print(f'--- {name} ({mask.sum()} paper) ---')
        print(f'   ...{sample[s:e]}...')
        print(f'   [match: {repr(m.group()[:60])}]\n')

### 3a. Pembersihan artefak
Yang dibuang **hanya** noise administratif. Tanda baca, stopword, kapitalisasi,
dan struktur kalimat **dipertahankan** — SBERT membutuhkannya.

In [ ]:
def clean_for_sbert(t):
    t = str(t)
    t = html.unescape(t)                                     # &amp; -> &
    # 1) perbaiki mojibake DULU (sebelum NFKC, karena NFKC mengubah ™ -> TM)
    for bad, good in [('â€™',"'"), ('â€˜',"'"), ('â€œ','"'), ('â€\x9d','"'),
                      ('â€"','—'), ('â€“','–'), ('Ã©','é'), ('Ã¨','è'),
                      ('Ã¼','ü'), ('Ã¶','ö'), ('Ã¡','á'), ('Â','')]:
        t = t.replace(bad, good)
    t = unicodedata.normalize('NFKC', t)                     # normalisasi unicode
    t = re.sub(r'<[/]?(i|b|sub|sup|em|italic|bold)>', '', t, flags=re.I)   # tag html
    t = re.sub(r'(?i)^\s*abstract\s*[—\-–:]\s*', '', t)                    # label Abstract—
    t = re.sub(r'(?i)(index terms|keywords)\s*[—\-–:].*$', '', t)          # blok keyword di akhir
    # 2) boilerplate copyright / publisher — tangkap kata 'copyright' sekalian
    t = re.sub(r'(?i)(copyright\s*)?(©|\(c\))\s*\d{4}[^.]*\.', ' ', t)
    t = re.sub(r'(?i)copyright\s*\d{4}[^.]*\.', ' ', t)
    t = re.sub(r'(?i)all rights reserved\.?', ' ', t)
    t = re.sub(r'(?i)published by [^.]*\.', ' ', t)
    t = re.sub(r'(?i)personal use is permitted[^.]*\.', ' ', t)
    t = re.sub(r'(?i)reprints? and permissions?[^.]*\.', ' ', t)
    t = re.sub(r'\[\d+(,\s*\d+)*\]', '', t)                                # sitasi [1], [2,3]
    t = re.sub(r'[\x00-\x08\x0b-\x1f\x7f]', ' ', t)                        # control chars
    # 3) rapikan spasi, termasuk spasi menggantung sebelum tanda baca
    t = re.sub(r'\s+([.,;:!?])', r'\1', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

df['text'] = df['text_raw'].apply(clean_for_sbert)
df['delta_chars'] = df['text_raw'].str.len() - df['text'].str.len()
print(f"Paper yang berubah setelah pembersihan: {(df.delta_chars>0).sum()} / {len(df)}")
print(f"Rata-rata karakter dibuang (yang berubah): {df.loc[df.delta_chars>0,'delta_chars'].mean():.1f}")

In [ ]:
# PREVIEW sebelum vs sesudah — ambil yang perubahannya paling besar
print('='*78)
print('PREVIEW PEMBERSIHAN (3 contoh dengan perubahan terbesar)')
print('='*78)
top = df.nlargest(3, 'delta_chars')
for i, r in top.iterrows():
    print(f"\n### Paper: {r['Document Title'][:70]}")
    print(f"    ({r['delta_chars']} karakter dibuang)")
    print('\n[ SEBELUM ]')
    print('  ' + r['text_raw'][:400].replace('\n',' ') + '...')
    print('\n[ SESUDAH ]')
    print('  ' + r['text'][:400] + '...')
    print('-'*78)

## 4. Statistik Panjang Title + Abstract
SPECTER2 memotong input di **512 token**. Karena 1 kata ≈ 1.3 token,
batas praktisnya sekitar **~390 kata**. Bagian yang melebihi akan hilang **tanpa peringatan**,
jadi kita ukur dulu seberapa besar risikonya.

In [ ]:
df['n_words'] = df['text'].str.split().str.len()
df['n_chars'] = df['text'].str.len()
df['est_tokens'] = (df['n_words'] * 1.3).round().astype(int)   # estimasi kasar

print('=== STATISTIK DESKRIPTIF: panjang title + abstract ===')
desc = df[['n_words','n_chars','est_tokens']].describe(
    percentiles=[.25,.5,.75,.90,.95,.99]).round(1)
print(desc)

W_LIMIT = int(MAX_TOKENS / 1.3)     # ~394 kata
print(f'\n=== RISIKO TRUNCATION (batas {MAX_TOKENS} token ≈ {W_LIMIT} kata) ===')
for lim, label in [(195,'256-token model'), (295,'384-token model'), (W_LIMIT,f'{MAX_TOKENS}-token (SPECTER2)')]:
    over = (df.n_words > lim).sum()
    print(f'  > {lim:>3} kata ({label:<22}): {over:>5} paper ({100*over/len(df):5.1f}%)')

trunc = (df.est_tokens > MAX_TOKENS).mean()*100
print(f'\nEstimasi paper terpotong di SPECTER2: {trunc:.1f}%')
print('  <5% -> aman;  5-15% -> masih wajar;  >15% -> pertimbangkan chunking')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13,4))
ax[0].hist(df.n_words, bins=60, color='#4C72B0', alpha=.85)
ax[0].axvline(W_LIMIT, ls='--', color='#C44E52', lw=2, label=f'batas ~{W_LIMIT} kata')
ax[0].set_title('Distribusi panjang (kata)'); ax[0].set_xlabel('jumlah kata'); ax[0].legend()

ax[1].boxplot(df.n_words, vert=False, widths=.6)
ax[1].axvline(W_LIMIT, ls='--', color='#C44E52', lw=2)
ax[1].set_title('Boxplot panjang (kata)'); ax[1].set_xlabel('jumlah kata')
plt.tight_layout(); plt.show()

## 5. Kalimat Pembanding (21 kalimat)
Diturunkan dari `LLM_SLR_Research_Plan.pptx` dan `Structured_IE_Reasoning-Augmented.pptx`.
Set: **A** (inti topik) + **B** (domain edukasi konkret) + **C** (tugas IE) + **kalimat 25** + **G** (human-in-the-loop).
Seluruh kalimat terikat eksplisit ke konteks edukasi.

In [ ]:
topic_sentences = [
    # --- A. Inti topik ---
    "This study uses large language models to extract information from educational data.",
    "We apply pre-trained language models for educational information extraction.",
    "Generative artificial intelligence is used to extract knowledge from educational content.",
    "A systematic approach to educational entity and attribute extraction using large language models.",
    "Large language models for extracting structured information in education.",
    # --- B. Domain edukasi konkret ---
    "Extracting information from student academic reports using language models.",
    "Using large language models to extract structured information from course syllabi.",
    "Automated extraction of insights from student feedback and course evaluations.",
    "Extracting information from e-learning discussion forums using LLMs.",
    "Language models for extracting metadata from exam questions and question banks.",
    "Extracting skills and competencies from student CVs for internship placement.",
    "Information extraction from study program accreditation reports using LLMs.",
    "Extracting learning analytics data from LMS and MOOC platforms using language models.",
    "Using LLMs to extract information from educational articles and reports.",
    "Building educational datasets through LLM-assisted information extraction.",
    # --- C. Tugas IE spesifik ---
    "Named entity recognition in educational text using large language models.",
    "Relation extraction from educational documents using pre-trained language models.",
    "Event extraction from academic records using generative language models.",
    "Constructing knowledge graphs from educational data using LLMs.",
    # --- Kalimat 25: kondisi data menantang ---
    "Information extraction from noisy and low-resource educational datasets.",
    # --- G. Human-in-the-loop ---
    "Human-in-the-loop annotation and educational dataset labeling with LLM feedback.",
]
print(f'{len(topic_sentences)} kalimat pembanding')

## 6. Encoding SBERT (SPECTER2) + Cosine Similarity
Model diunduh sekali (~440 MB) lalu di-cache. Encoding ribuan abstrak di CPU
memakan beberapa menit; jika ada GPU akan otomatis dipakai.

`normalize_embeddings=True` diperlukan agar cosine similarity valid.

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model = SentenceTransformer(MODEL_NAME, device=device)
model.max_seq_length = MAX_TOKENS
print(f'Model: {MODEL_NAME} | max_seq_length={model.max_seq_length}')

In [ ]:
%%time
# encode dokumen (batch) dan kalimat pembanding
doc_emb = model.encode(df['text'].tolist(), batch_size=32,
                       show_progress_bar=True, normalize_embeddings=True,
                       convert_to_numpy=True)
qry_emb = model.encode(topic_sentences, batch_size=8,
                       normalize_embeddings=True, convert_to_numpy=True)
print('doc_emb:', doc_emb.shape, '| qry_emb:', qry_emb.shape)

In [ ]:
sim_matrix = cosine_similarity(doc_emb, qry_emb)     # (n_docs, 21)
df['sim_max']  = sim_matrix.max(axis=1)
df['sim_mean'] = sim_matrix.mean(axis=1)
df['best_sentence_idx'] = sim_matrix.argmax(axis=1)

print('=== DISTRIBUSI SKOR SEMANTIC (SPECTER2) ===')
print(df[['sim_max','sim_mean']].describe(
    percentiles=[.25,.5,.75,.90,.95,.99]).round(4))
print('\nCatatan: skala skor SBERT BERBEDA TOTAL dari TF-IDF.')
print('Threshold lama (mis. 0.11) TIDAK berlaku — dihitung ulang di bawah.')

## 7. Penentuan Threshold — Data-Driven
Fungsi yang sama seperti notebook sebelumnya (Kneedle + Otsu + p90, diambil **median**
sebagai konsensus), tetapi dihitung ulang dari nol pada distribusi SBERT.

Karena embedding SBERT bersifat *dense*, skornya cenderung menumpuk pada rentang sempit
dan elbow-nya bisa kurang tajam dibanding TF-IDF — cek plot untuk memastikan.

In [ ]:
def data_driven_threshold(scores, S=1.0, verbose=True):
    scores = np.asarray(scores)

    # Kneedle
    s = np.sort(scores)[::-1]; x = np.arange(len(s))
    kl = KneeLocator(x, s, curve='convex', direction='decreasing', S=S)
    knee_i = kl.knee if kl.knee is not None else int(np.percentile(x, 90))
    thr_knee = s[knee_i]

    # Otsu
    hist, edges = np.histogram(scores, bins=256)
    p = hist/hist.sum(); centers = (edges[:-1]+edges[1:])/2
    best_t, best_v = centers[0], -1
    for i in range(1, 256):
        w0, w1 = p[:i].sum(), p[i:].sum()
        if w0 == 0 or w1 == 0: continue
        m0 = (p[:i]*centers[:i]).sum()/w0; m1 = (p[i:]*centers[i:]).sum()/w1
        v = w0*w1*(m0-m1)**2
        if v > best_v: best_v, best_t = v, centers[i]

    thr_p90 = np.percentile(scores, 90)
    res = {'kneedle': thr_knee, 'otsu': best_t, 'p90': thr_p90}
    res['CONSENSUS(median)'] = float(np.median(list(res.values())))

    if verbose:
        for k, v in res.items():
            mark = ' <== dipakai' if k.startswith('CONSENSUS') else ''
            print(f'  {k:18s}: thr={v:.4f} -> accepted {(scores>=v).sum()}{mark}')
        base = [res['kneedle'], res['otsu'], res['p90']]
        spread = max(base)-min(base)
        print(f'  --> spread = {spread:.4f} '
              f'({"KONVERGEN (kuat)" if spread<0.05 else "moderat — median menstabilkan"})')
    return res

print('=== Threshold semantic (SPECTER2, sim_max) ===')
th = data_driven_threshold(df['sim_max'].values)
SEM_THRESH = th['CONSENSUS(median)']
print(f'\nSEM_THRESH = {SEM_THRESH:.4f}')

### 7a. Uji pembuktian threshold

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16,4))

ax[0].hist(df.sim_max, bins=70, color='#4C72B0', alpha=.85)
for k, v in th.items():
    ax[0].axvline(v, ls='--', lw=1.4, label=f'{k.split("(")[0]}={v:.3f}')
ax[0].set_title('Distribusi sim_max + threshold'); ax[0].set_xlabel('cosine (SBERT)'); ax[0].legend(fontsize=8)

s = np.sort(df.sim_max)[::-1]
ax[1].plot(s, color='#C44E52')
ki = np.argmin(np.abs(s - SEM_THRESH))
ax[1].axvline(ki, ls='--', color='k', label=f'threshold @rank {ki}')
ax[1].set_title('Kurva skor terurut (elbow)'); ax[1].set_xlabel('rank'); ax[1].legend(fontsize=8)

grid = np.linspace(df.sim_max.min(), df.sim_max.max(), 100)
ax[2].plot(grid, [(df.sim_max>=g).sum() for g in grid], color='#55A868')
ax[2].axvline(SEM_THRESH, ls='--', color='k', label=f'chosen={SEM_THRESH:.3f}')
ax[2].set_title('Sensitivity: #accepted vs threshold'); ax[2].set_xlabel('threshold'); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Inspeksi kualitatif: paper tepat di atas vs di bawah threshold
print('--- 5 paper TEPAT DI ATAS threshold ---')
for _, r in df[df.sim_max>=SEM_THRESH].nsmallest(5,'sim_max').iterrows():
    print(f'  + [{r.sim_max:.3f}] {r["Document Title"][:80]}')
print('\n--- 5 paper TEPAT DI BAWAH threshold ---')
for _, r in df[df.sim_max<SEM_THRESH].nlargest(5,'sim_max').iterrows():
    print(f'  - [{r.sim_max:.3f}] {r["Document Title"][:80]}')

print('\n--- 5 paper skor TERTINGGI (sanity check) ---')
for _, r in df.nlargest(5,'sim_max').iterrows():
    print(f'  * [{r.sim_max:.3f}] {r["Document Title"][:80]}')
    print(f'      cocok dgn: "{topic_sentences[r.best_sentence_idx][:70]}..."')

In [ ]:
# Kalimat pembanding mana yang paling sering jadi 'best match'?
vc = df.loc[df.sim_max>=SEM_THRESH, 'best_sentence_idx'].value_counts()
print('=== Kontribusi tiap kalimat pembanding (paper accepted) ===')
for idx, n in vc.items():
    print(f'  [{n:>4}] {topic_sentences[idx][:78]}')
unused = set(range(len(topic_sentences))) - set(vc.index)
if unused:
    print('\nKalimat yang TIDAK pernah jadi best match (kandidat dibuang):')
    for i in unused: print(f'  - {topic_sentences[i][:78]}')

## 8. Export

In [ ]:
accepted = df[df['sim_max'] >= SEM_THRESH].copy()
accepted = accepted.drop_duplicates(subset=['Document Title','Abstract']).reset_index(drop=True)
accepted['method'] = 'sbert_specter2'

out_cols = ['ID','Document Title','Publication Title','Publication Year','Abstract',
            'Author Keywords','Document Identifier','sim_max','sim_mean','method']
out_cols = [c for c in out_cols if c in accepted.columns]
accepted[out_cols].to_csv(OUT_PATH, index=False)

print(f'Total accepted : {len(accepted)}  -> {OUT_PATH}')
print('\n=== Document Identifier (accepted) ===')
print(accepted['Document Identifier'].value_counts())

In [ ]:
# Kandidat validasi manual: zona ambang ±10% threshold
lo, hi = SEM_THRESH*0.9, SEM_THRESH*1.1
band = df[(df.sim_max>=lo) & (df.sim_max<=hi)].copy()
band['side'] = np.where(band.sim_max>=SEM_THRESH, 'above', 'below')
band = band.sort_values('sim_max', ascending=False)
band['relevant_manual'] = ''      # isi 1 = relevan, 0 = tidak
band['note'] = ''

vcols = [c for c in ['ID','Document Title','Publication Title','Publication Year','Abstract',
                     'Author Keywords','Document Identifier','sim_max','side',
                     'relevant_manual','note'] if c in band.columns]
band[vcols].to_csv(VAL_OUT, index=False)
print(f'{len(band)} kandidat validasi manual -> {VAL_OUT}')
print(band['side'].value_counts())

---
### Catatan metodologi

**Preprocessing untuk SBERT berbeda dari TF-IDF.** Stopword, tanda baca, kapitalisasi, dan
struktur kalimat dipertahankan karena transformer memanfaatkannya untuk memahami makna
(mis. negasi *"not"* dan relasi *"for"* vs *"from"*). Stemming/lemmatization juga tidak dipakai
karena SPECTER2 memakai tokenizer subword. Yang dibersihkan hanya artefak teknis.

**Asumsi yang dipenuhi:**
1. Query dan dokumen sebanding — kalimat pembanding ditulis sebagai kalimat natural, bukan frasa keyword.
2. Domain model cocok — SPECTER2 dilatih pada paper akademik.
3. Skala skor dihitung ulang — threshold TF-IDF tidak dipakai kembali.
4. Vektor dinormalisasi (`normalize_embeddings=True`).
5. Panjang input diperiksa terhadap batas 512 token.

**Referensi:**
- Cohan et al. (2020). *SPECTER: Document-level Representation Learning using Citation-informed Transformers.* ACL.
- Reimers & Gurevych (2019). *Sentence-BERT.* EMNLP.
- Satopää et al. (2011). *Finding a "Kneedle" in a Haystack.* IEEE ICDCS Workshops.
- Otsu (1979). *A Threshold Selection Method from Gray-Level Histograms.* IEEE TSMC.